# Rare-Event Probability Tables — multi-ticker

Tickers and their per-ticker parameters come from `configs/tickers.yaml`; delete
that file and the run falls back to ES=F, NQ=F, YM=F, RTY=F.

For each ticker: the rare-event table and the streak-frequency chart. Then two
cross-ticker summaries.


In [ ]:
# ── Environment setup: Google Colab vs local ────────────────────────
# Locally the project is installed once from the repo root with
# `pip install -e .`, so `from src.tools...` resolves from any working
# directory. On Colab the same editable install is done here against the
# copy of the repo on Drive.
import os

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    !pip install -q -e /content/drive/MyDrive/github/quant_dev

print(f'Running in Colab: {IN_COLAB}')

In [ ]:
from IPython.display import display

from src.tools.price_return import (
    load_ticker_config, analyze_ticker, compare_tickers,
    format_probability_table, plot_streak_frequency,
)

## Configuration


In [ ]:
# Tickers and their parameters come from configs/tickers.yaml. Pass an explicit
# path to load_ticker_config(...) to use a different file.
CONFIG = load_ticker_config()

DRILL_N_DAYS = 3        # holding period drilled into by the rare-event tables

## Per-ticker analysis


In [ ]:
# One bad symbol should not abort the whole run, so failures are collected and
# reported rather than raised. yfinance surfaces more than just ValueError.
results, failures = {}, {}

for symbol, P in CONFIG.items():
    try:
        results[symbol] = analyze_ticker(P, drill_n_days=DRILL_N_DAYS)
    except Exception as exc:
        failures[symbol] = exc
        print(f'SKIPPED {symbol}: {type(exc).__name__}: {exc}')
        continue

    r = results[symbol]
    print(f'\n{"=" * 78}\n{P.label}  ({symbol})   '
          f'{len(r["df"])} trading days, threshold +/-{P.win_threshold}%\n')
    print(f'{len(r["drill"])} low-probability events '
          f'for a {DRILL_N_DAYS}-day holding period')
    display(format_probability_table(r['drill']))
    plot_streak_frequency(r['df'], r['streaks'], P).show()

if not results:
    raise RuntimeError('Every ticker failed - check the config and network.')

## Cross-ticker summaries


In [ ]:
streak_tbl, dist_tbl = compare_tickers(results, ratio_window=DRILL_N_DAYS)

print('Streak frequency (% of all trading days), win / loss')
display(streak_tbl)

print('\nReturn distribution - streaks count threshold-clearing days, so they '
      'track the median rather than the mean; skew separates the two.')
display(dist_tbl)

if failures:
    print('\nFailed: ' + ', '.join(failures))